# Обучение «полной мощью» на Kaggle

Конфигурация: **ResNet50, 512×512, BCE+Dice, TTA** — под лимит 100 GFLOPs (85.5 GFLOPs).

## Что сделать перед запуском

1. Загрузи на Kaggle как **датасет** папки `train_stage1` и `test_stage1` (сохранив вложенность
   как на диске: `train_stage1/stage1/...` и `test_stage1/test_stage1/...`).
2. Укажи имя датасета в `DATA_ROOT` ниже.
3. Код клонируется из GitHub — если репозиторий приватный, сделай его временно публичным
   (Settings → General → Danger Zone → Change visibility) или загрузи папку `src/` вручную.
4. Запусти все ячейки (Runtime → Run all).

In [ ]:
import subprocess, sys

def have_cuda_torch():
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False

# torch ставим только если в окружении нет CUDA-версии (иначе pip долго переустанавливает)
if not have_cuda_torch():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.5.1+cu121", "torchvision==0.20.1+cu121",
                    "--index-url", "https://download.pytorch.org/whl/cu121"])

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "segmentation-models-pytorch==0.4.0", "opencv-python-headless",
                "Pillow", "numpy==1.26.4", "tqdm", "matplotlib", "pandas"])

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))

In [ ]:
import sys
from pathlib import Path

# Имя Kaggle-датасета с данными конкурса
DATA_ROOT = "/kaggle/input/ai-challenge-data"

# Клонируем код решения
import subprocess
subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/ssaikk25/cv.git", "/kaggle/working/repo"])
sys.path.insert(0, "/kaggle/working/repo")

# Параметры обучения
params = dict(
    encoder="resnet50",
    img_size=512,
    batch_size=16,
    epochs=15,
    dice_w=1.0,
    neg_ratio=0.2,
    val_neg=1500,
    name="manip_resnet50_512",
)

TRAIN_DIR = f"{DATA_ROOT}/train_stage1"
TRAIN_CSV = f"{TRAIN_DIR}/stage1/train.csv"
TEST_DIR = f"{DATA_ROOT}/test_stage1/test_stage1"
TEST_CSV = f"{TEST_DIR}/test.csv"

print("TRAIN_CSV:", Path(TRAIN_CSV).exists(), TRAIN_CSV)
print("TEST_CSV :", Path(TEST_CSV).exists(), TEST_CSV)

## 1. Кэш статистики масок

Определяет, какие строки — позитивы, а какие — чистые (негативы для FPR).

In [ ]:
subprocess.run([sys.executable, "-m", "src.prepare",
                "--data-dir", TRAIN_DIR, "--train-csv", TRAIN_CSV,
                "--output", "/kaggle/working/mask_stats.json"],
               cwd="/kaggle/working/repo")

## 2. Обучение

Чекпоинты и подобранный порог сохраняются в `/kaggle/working/checkpoints/`.

In [ ]:
cmd = [sys.executable, "-m", "src.train",
       "--data-dir", TRAIN_DIR,
       "--train-csv", TRAIN_CSV,
       "--mask-stats", "/kaggle/working/mask_stats.json",
       "--img-size", str(params["img_size"]),
       "--batch-size", str(params["batch_size"]),
       "--epochs", str(params["epochs"]),
       "--dice-w", str(params["dice_w"]),
       "--neg-ratio", str(params["neg_ratio"]),
       "--val-neg", str(params["val_neg"]),
       "--encoder", params["encoder"],
       "--num-workers", "2",
       "--checkpoint-dir", "/kaggle/working/checkpoints",
       "--name", params["name"]]
subprocess.run(cmd, cwd="/kaggle/working/repo")

## 3. Инференс с TTA и сборка посылки

In [ ]:
cmd = [sys.executable, "-m", "src.predict",
       "--data-dir", TEST_DIR,
       "--test-csv", TEST_CSV,
       "--encoder", params["encoder"],
       "--img-size", str(params["img_size"]),
       "--checkpoint", f"/kaggle/working/checkpoints/{params['name']}_best.pth",
       "--threshold-json", f"/kaggle/working/checkpoints/{params['name']}_threshold.json",
       "--tta",
       "--num-workers", "2",
       "--pred-dir", "/kaggle/working/predictions",
       "--submission-csv", "/kaggle/working/submission.csv",
       "--zip", "/kaggle/working/submission.zip"]
subprocess.run(cmd, cwd="/kaggle/working/repo")

## 4. Готово

Скачай `/kaggle/working/submission.zip` через панель **Output** на Kaggle и загрузи на платформу конкурса.

Если время сессии ограничено — уменьши `epochs` (например до 8) или `img_size` до 384.